# SoundScape Band simulation

Run the same Python acoustic simulation and DEDA policy used by the local browser lab.

**Scope:** synthetic free-field microphone signals and simulated class probabilities. No trained audio classifier, physical wearable or real motor latency is measured. New acoustic-suite results are separate from the paper's decision-stream benchmark.

Before starting, run `python python/package_simulation.py` on your computer. This creates a source ZIP in `results/simulation/`. Upload that ZIP in the next cell. Do not upload private recordings or unrelated folders.


In [ ]:
# Colab only: select the source ZIP created by package_simulation.py.
from google.colab import files
from pathlib import Path
from zipfile import ZipFile
import io

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Select exactly one SoundScape source ZIP')
archive_name, archive_bytes = next(iter(uploaded.items()))
workspace = Path('/content/soundscape_source')
workspace.mkdir(exist_ok=True)
with ZipFile(io.BytesIO(archive_bytes)) as archive:
    for name in archive.namelist():
        target = (workspace / name).resolve()
        if not target.is_relative_to(workspace.resolve()):
            raise ValueError('Unsafe path in source archive')
    archive.extractall(workspace)
ROOT = workspace / 'soundscape-band'
if not (ROOT / 'python/deda/simulator.py').exists():
    raise ValueError('This ZIP does not contain the connected SoundScape simulator')
print('Source loaded:', ROOT)


In [ ]:
%pip install -q numpy matplotlib pytest
import sys
sys.path.insert(0, str(ROOT / 'python'))
from deda.simulator import simulate, experiment
from run_simulator_experiment import save_results
import json
import numpy as np
import matplotlib.pyplot as plt
print('Python', sys.version.split()[0], '| NumPy', np.__version__)


## One connected acoustic scenario

Change `sound`, true `angle`, `snr_db`, `condition` and `confidence`. Angles run clockwise from front: 0° front, 90° right, 180° back, 270° left. The algorithm estimates the angle independently.

Conditions: `clean`, `interference`, `clock_fault`, `reflection`, `dropout`. Dropout explicitly removes direction observations; it is a fault injection. Confidence is scripted input and is independent of acoustic noise.


In [ ]:
scene = simulate(dict(sound='horn', angle=180, snr_db=20,
                      condition='clean', confidence=0.95, seed=7))
print(json.dumps({'DEDA': scene['deda'], 'Baseline B': scene['baseline']}, indent=2))


In [ ]:
frames = scene['frames']
times = np.array([f['t_ms'] for f in frames])
active = np.array([f['source_active'] for f in frames])
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(times[active], [f['direction']['theta_deg'] for f in frames if f['source_active']], 'o-', label='Estimated direction')
axes[0].axhline(scene['options']['angle'], ls='--', color='green', label='True source direction')
axes[0].set(ylabel='Angle (degrees)', ylim=(-5,365))
axes[0].legend()
axes[1].plot(times, [f['direction']['coherence'] for f in frames], label='Minimum PHAT coherence')
axes[1].axhline(0.30, ls='--', color='gray', label='Per-window acceptance floor')
for a in scene['alerts']:
    axes[1].axvline(a['t_ms'], color='green', alpha=.6)
    axes[1].text(a['t_ms'], .9, a['kind'], fontsize=8)
axes[1].set(xlabel='Simulated time (ms)', ylabel='Coherence', ylim=(0,1.05))
axes[1].legend()
fig.suptitle('Synthetic microphone signals → GCC-PHAT → actual DEDA policy')
fig.tight_layout()
plt.show()


## Reproducible acoustic stress suite

Start with `quick` and two seeds (64 critical trials + two short ambient controls). Full coverage has 192 critical trials per seed. Both use frozen, uncalibrated default policies. Scripted class probabilities vary among 65%, 80% and 95%; ambient inputs have no false confusers.

Compare wrong-direction errors, misses and general warnings together. This suite is not a reproduction of the paper's calibrated decision-stream experiment and does not establish real classifier performance.


In [ ]:
results = experiment({'suite': 'quick', 'seeds': 2})
print(json.dumps(results['policies'], indent=2))
output = save_results(results, ROOT / 'results/simulation/notebook_run')
from IPython.display import Image, display
display(Image(filename=str(output / 'comparison.png')))
print(results['note'])


## Verify the source implementation

This optional cell builds the desktop C core and runs both the existing tests and connected-simulation tests. It does not compile for a microcontroller.


In [ ]:
import subprocess
subprocess.run([sys.executable, str(ROOT / 'python/build_c.py')], check=True)
import os
test_env = {**os.environ, 'PYTEST_DISABLE_PLUGIN_AUTOLOAD': '1'}
subprocess.run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', 'tests'],
               cwd=ROOT / 'python', env=test_env, check=True)


## Download the numerical results

The ZIP contains JSON with trial settings and outcomes, CSV, a comparison graph and explanatory notes. Do not label simulated alert delay as measured motor latency.


In [ ]:
import shutil
scene_file = output / 'single_scene.json'
scene_file.write_text(json.dumps(scene, indent=2, allow_nan=False))
archive = shutil.make_archive('/content/soundscape-results', 'zip', output)
files.download(archive)


## Original paper reproduction and future extensions

The existing `python/acoustic_checks.py` and `python/run_synthetic.py` reproduce the original acoustic checks and decision-level policy benchmark. The latter calibrates thresholds on separate streams, sweeps many settings and can take considerably longer. Run it separately and preserve its existing results before overwriting them.

Next research stages are recorded source audio, a trained classifier with independent evaluation clips, realistic room/body acoustics and eventual hardware/user testing. The local browser application is the presentation interface; this notebook is a portable numerical workflow.
